In [1]:
import pandas as pd
import numpy as np
from functions.running import prepare_data
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.decomposition import PCA
from functions.training import train
from functions.networks import SimpleNN, FullNN
from numpy.linalg import svd


In [2]:
import numpy as np
from numpy.linalg import svd

import numpy as np

def principal_angles(X, Y, eps=1e-10):
    """
    Compute the principal angles between subspaces X and Y using the
    recursive definition (Björck & Golub, 1973).
    
    Parameters
    ----------
    X : np.ndarray of shape (n, l)
        Orthonormal basis of subspace X (columns are vectors)
    Y : np.ndarray of shape (n, s)
        Orthonormal basis of subspace Y (columns are vectors)
    
    Returns
    -------
    angles_deg : np.ndarray
        Principal angles in degrees
    """
    # Orthonormalize X and Y just in case
    Qx, _ = np.linalg.qr(X)
    Qy, _ = np.linalg.qr(Y)
    
    # Compute the SVD of Qx^T Qy
    U, s, Vh = np.linalg.svd(Qx.T @ Qy)
    
    # Clip for numerical stability
    s = np.clip(s, -1.0, 1.0)
    
    # Principal angles in radians
    angles_rad = np.arccos(s)
    
    # Convert to degrees
    angles_deg = np.degrees(angles_rad)
    
    return angles_rad, angles_deg


def get_weights(X_train, y_train, X_test, y_test, batch_size=64, n_layer=5, variance_retained=0.95, 
                init_type='he', learning_rate = 0.01, n_frozen_epochs=30, epochs = 200):
    input_dim = X_train.shape[1]  # Number of features

    output_dim = len(np.unique(y_train))  # Number of classes
    pca = PCA(n_components=variance_retained)
    pca.fit(X_train)
    n_components = pca.n_components_
    print(n_components)
    hidden_dim = n_components  # Hidden layer size

    other_layers = SimpleNN(input_dim=n_components, hidden_dim=hidden_dim, output_dim=output_dim, n_layer = n_layer, init_type = init_type)

    train_loader = torch.utils.data.DataLoader(list(zip(X_train, y_train)), batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(list(zip(X_test, y_test)), batch_size=batch_size, shuffle=False)

    criterion = nn.CrossEntropyLoss()
    

    # *********** Neural Network with PCA-initialized weights *******************************************************************
    print("Training with PCA-initialized NN...")
    pca_init_nn = FullNN(input_dim, n_components, other_layers, activation='none', init_type=init_type)
    pca_init_nn.init_pca_weights(X_train)  # Initialize weights with PCA components
    W_r = pca_init_nn.fc1.weight.detach().cpu().numpy().copy()

    # train on everything except the first layer
    optimizer = optim.Adam([{'params': param} for name, param in pca_init_nn.named_parameters() if not name.startswith('fc1')],
                            lr=learning_rate)
    model1 = train(pca_init_nn, train_loader, test_loader, criterion, optimizer, epochs=n_frozen_epochs)[0]
    # train the complete network
    optimizer = optim.Adam(pca_init_nn.parameters(), lr=learning_rate)
    model2 = train(pca_init_nn, train_loader, test_loader, criterion, optimizer, epochs=epochs-n_frozen_epochs)[0]
    W_optimal = model2.fc1.weight.detach().cpu().numpy()

    return W_r, W_optimal



### Heart dataset

In [3]:
df_train = pd.read_table('https://archive.ics.uci.edu/ml/machine-learning-databases/spect/SPECTF.train', header = None,sep=',')
df_test = pd.read_table('https://archive.ics.uci.edu/ml/machine-learning-databases/spect/SPECTF.test',
                     header=None, sep = ',')
df = pd.concat([df_train, df_test])
data = df.to_numpy()
X,y = data[:,1:], data[:,0]
G = len(np.unique(y))
for g in range(G):
  print(sum(y==g))
X = X.astype('float')

missing = False

X_train, X_test, y_train, y_test = prepare_data(X,y, missing = missing)

55
212


In [4]:
W_r, W_optimal = get_weights(X_train, y_train, X_test, y_test)
angle_rad, angle_deg = principal_angles(W_r.T, W_optimal.T)

print("Principal angles (deg):", angle_deg)
print("Mean principal angle (deg):", angle_deg.mean())
print("Cosine similarity:", np.cos(angle_rad).mean())

23
Training with PCA-initialized NN...
Number of PCA components: 23
Epoch 1/30, Training Loss: 0.6920, Testing Accuracy: 0.8148, Training Time: 0.0105
Epoch 2/30, Training Loss: 0.5189, Testing Accuracy: 0.8148, Training Time: 0.0204
Epoch 3/30, Training Loss: 0.4875, Testing Accuracy: 0.8148, Training Time: 0.0270
Epoch 4/30, Training Loss: 0.4536, Testing Accuracy: 0.8148, Training Time: 0.0338
Epoch 5/30, Training Loss: 0.4404, Testing Accuracy: 0.8148, Training Time: 0.0403
Epoch 6/30, Training Loss: 0.4284, Testing Accuracy: 0.8148, Training Time: 0.0473
Epoch 7/30, Training Loss: 0.4124, Testing Accuracy: 0.8148, Training Time: 0.0543
Epoch 8/30, Training Loss: 0.3978, Testing Accuracy: 0.8148, Training Time: 0.0623
Epoch 9/30, Training Loss: 0.3821, Testing Accuracy: 0.8148, Training Time: 0.0689
Epoch 10/30, Training Loss: 0.3592, Testing Accuracy: 0.8148, Training Time: 0.0759
Epoch 11/30, Training Loss: 0.3351, Testing Accuracy: 0.8642, Training Time: 0.0825
Epoch 12/30, Trai

### Parkinson dataset

In [5]:
data = pd.read_csv('/Volumes/Macintosh HD/Projects/PAPER/12.PCSINIT/data/pd_speech_features.csv', sep = ",", header = [0,1])
print(data.shape)
data.head()
data = data.to_numpy()
X, y = data[:,:-1], data[:,-1]
G = len(np.unique(y))
print('input shape:', X.shape)

missing = False

X_train, X_test, y_train, y_test = prepare_data(X,y, missing = missing)

(756, 755)
input shape: (756, 754)


In [6]:
W_r, W_optimal = get_weights(X_train, y_train, X_test, y_test)
angle_rad, angle_deg = principal_angles(W_r.T, W_optimal.T)

print("Principal angles (deg):", angle_deg)
print("Mean principal angle (deg):", angle_deg.mean())
print("Cosine similarity:", np.cos(angle_rad).mean())

151
Training with PCA-initialized NN...
Number of PCA components: 151
Epoch 1/30, Training Loss: 2.4373, Testing Accuracy: 0.7269, Training Time: 0.0523
Epoch 2/30, Training Loss: 0.5389, Testing Accuracy: 0.7313, Training Time: 0.1023
Epoch 3/30, Training Loss: 0.4561, Testing Accuracy: 0.7357, Training Time: 0.1409
Epoch 4/30, Training Loss: 0.3497, Testing Accuracy: 0.7665, Training Time: 0.1753
Epoch 5/30, Training Loss: 0.2604, Testing Accuracy: 0.7930, Training Time: 0.2078
Epoch 6/30, Training Loss: 0.2017, Testing Accuracy: 0.7533, Training Time: 0.2385
Epoch 7/30, Training Loss: 0.1792, Testing Accuracy: 0.7841, Training Time: 0.2692
Epoch 8/30, Training Loss: 0.0905, Testing Accuracy: 0.7974, Training Time: 0.3009
Epoch 9/30, Training Loss: 0.0911, Testing Accuracy: 0.7841, Training Time: 0.3336
Epoch 10/30, Training Loss: 0.1167, Testing Accuracy: 0.7621, Training Time: 0.3674
Epoch 11/30, Training Loss: 0.1625, Testing Accuracy: 0.7753, Training Time: 0.4066
Epoch 12/30, Tr

### HTDA dataset

### Micromass dataset

### 

In [7]:
from sklearn.preprocessing import LabelEncoder

label_data = pd.read_csv('/Volumes/Macintosh HD/Projects/PAPER/12.PCSINIT/data/micromass/mixed_spectra_metadata.csv', sep = ';')
label = label_data[['Mixture_Label']]
label = label.to_numpy()
print(len(np.unique(label)))
le2 = LabelEncoder()
y = le2.fit_transform(label)

df = pd.read_csv('/Volumes/Macintosh HD/Projects/PAPER/12.PCSINIT/data/micromass/mixed_spectra_matrix.csv', sep = ';', header = None)
print(df.shape)
print(df.head())
X = df.to_numpy()
var_vec = np.array([np.var(X[:,i]) for i in range(X.shape[1])])
id = np.where(var_vec > 1e-5)
X = X[:,id].reshape((len(X),-1))
X.shape

missing = False

X_train, X_test, y_train, y_test = prepare_data(X,y, missing = missing)

10
(360, 1300)
   0     1     2     3     4     5     6     7     8             9     ...  \
0     0   0.0   0.0   0.0   0.0     0   0.0   0.0   0.0      0.000000  ...   
1     0   0.0   0.0   0.0   0.0     0   0.0   0.0   0.0  10162.613281  ...   
2     0   0.0   0.0   0.0   0.0     0   0.0   0.0   0.0      0.000000  ...   
3     0   0.0   0.0   0.0   0.0     0   0.0   0.0   0.0      0.000000  ...   
4     0   0.0   0.0   0.0   0.0     0   0.0   0.0   0.0  15409.125977  ...   

   1290  1291  1292  1293  1294  1295          1296  1297  1298          1299  
0   0.0     0     0   0.0   0.0   0.0      0.000000   0.0   0.0      0.000000  
1   0.0     0     0   0.0   0.0   0.0  11427.124023   0.0   0.0      0.000000  
2   0.0     0     0   0.0   0.0   0.0  12094.415039   0.0   0.0      0.000000  
3   0.0     0     0   0.0   0.0   0.0      0.000000   0.0   0.0      0.000000  
4   0.0     0     0   0.0   0.0   0.0      0.000000   0.0   0.0  35418.402344  

[5 rows x 1300 columns]


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [8]:
W_r, W_optimal = get_weights(X_train, y_train, X_test, y_test)
angle_rad, angle_deg = principal_angles(W_r.T, W_optimal.T)

print("Principal angles (deg):", angle_deg)
print("Mean principal angle (deg):", angle_deg.mean())
print("Cosine similarity:", np.cos(angle_rad).mean())

145
Training with PCA-initialized NN...
Number of PCA components: 145
Epoch 1/30, Training Loss: 4.4891, Testing Accuracy: 0.2870, Training Time: 0.0429
Epoch 2/30, Training Loss: 1.9421, Testing Accuracy: 0.4074, Training Time: 0.0930
Epoch 3/30, Training Loss: 1.2491, Testing Accuracy: 0.5741, Training Time: 0.1113
Epoch 4/30, Training Loss: 0.6288, Testing Accuracy: 0.6944, Training Time: 0.1301
Epoch 5/30, Training Loss: 0.2908, Testing Accuracy: 0.7685, Training Time: 0.1445
Epoch 6/30, Training Loss: 0.1903, Testing Accuracy: 0.8426, Training Time: 0.1649
Epoch 7/30, Training Loss: 0.0552, Testing Accuracy: 0.7593, Training Time: 0.1825
Epoch 8/30, Training Loss: 0.0484, Testing Accuracy: 0.8426, Training Time: 0.2013
Epoch 9/30, Training Loss: 0.0215, Testing Accuracy: 0.8241, Training Time: 0.2190
Epoch 10/30, Training Loss: 0.0859, Testing Accuracy: 0.8241, Training Time: 0.2396
Epoch 11/30, Training Loss: 0.0847, Testing Accuracy: 0.8148, Training Time: 0.2611
Epoch 12/30, Tr

### Ionosphere dataset

In [9]:
from sklearn.preprocessing import LabelEncoder
data = pd.read_csv('http://archive.ics.uci.edu/ml/machine-learning-databases/ionosphere/ionosphere.data',
                  sep = ",", header = None)
data = pd.DataFrame.to_numpy(data)
X, y = data[:,:34].astype(np.float64), data[:,34]
le2 = LabelEncoder()
y = le2.fit_transform(y)
G = len(np.unique(y))
X = np.delete(X,[0,1], axis = 1)
for g in range(G):
  print(sum(y==g))

missing = False

X_train, X_test, y_train, y_test = prepare_data(X,y, missing = missing)

126
225


In [10]:
W_r, W_optimal = get_weights(X_train, y_train, X_test, y_test)
angle_rad, angle_deg = principal_angles(W_r.T, W_optimal.T)

print("Principal angles (deg):", angle_deg)
print("Mean principal angle (deg):", angle_deg.mean())
print("Cosine similarity:", np.cos(angle_rad).mean())

22
Training with PCA-initialized NN...
Number of PCA components: 22
Epoch 1/30, Training Loss: 0.8848, Testing Accuracy: 0.6887, Training Time: 0.0393
Epoch 2/30, Training Loss: 0.5186, Testing Accuracy: 0.8208, Training Time: 0.0500
Epoch 3/30, Training Loss: 0.3786, Testing Accuracy: 0.8491, Training Time: 0.0603
Epoch 4/30, Training Loss: 0.2676, Testing Accuracy: 0.8774, Training Time: 0.0705
Epoch 5/30, Training Loss: 0.2051, Testing Accuracy: 0.9151, Training Time: 0.0821
Epoch 6/30, Training Loss: 0.1417, Testing Accuracy: 0.9245, Training Time: 0.0955
Epoch 7/30, Training Loss: 0.0927, Testing Accuracy: 0.9245, Training Time: 0.1058
Epoch 8/30, Training Loss: 0.0760, Testing Accuracy: 0.9340, Training Time: 0.1154
Epoch 9/30, Training Loss: 0.0491, Testing Accuracy: 0.9434, Training Time: 0.1256
Epoch 10/30, Training Loss: 0.0381, Testing Accuracy: 0.9528, Training Time: 0.1406
Epoch 11/30, Training Loss: 0.0337, Testing Accuracy: 0.9528, Training Time: 0.1528
Epoch 12/30, Trai